# 02.06 Prepared Dataset Validation

**Phase 02 — Data Preparation**  
*OFPHIL V2: Charity Patient Eligibility Classification*

---

### Description & Purpose
Perform final validation and quality assurance of prepared datasets before feature engineering and model training splits.


### Research Objectives

1. Validate schema conformance, shape, non-null guarantees, and data types of prepared data.
2. Verify that no direct patient identifiers (names, addresses, contact details) remain.
3. Confirm that the target variable (charity eligibility class) is appropriately isolated and encoded.
4. Generate final validation certification for Phase 02 completion.


### Inputs & Expected Outputs

**Required Inputs:**
- `Cleaned and encoded dataset artifacts from Phase 02 (`data/interim/cleaned/`).`

**Expected Outputs:**
- `Prepared dataset validation report: `data/interim/validated/prepared_dataset_validation_report.json`.`
- `Verified tabular dataset ready for feature engineering.`


### 1. Environment Setup & Library Imports

In [ ]:
from pathlib import Path
import json, datetime, re
import pandas as pd
import numpy as np

def _find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'data').exists() and (p / 'experiments').exists():
            return p
    return Path('../../..').resolve()

PROJECT_ROOT = _find_project_root(Path.cwd())

### 1. Setup Environment and Load Prepared Datasets

In [ ]:
CLEANED_CSV = PROJECT_ROOT / 'data' / 'interim' / 'cleaned' / 'ofphil_cleaned_tabular.csv'
SPLITS_DIR = PROJECT_ROOT / 'data' / 'processed' / 'splits'
VALIDATED_DIR = PROJECT_ROOT / 'data' / 'interim' / 'validated'
VALIDATED_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_CSV_PATH = PROJECT_ROOT / 'data' / 'processed' / 'validation' / 'prepared_dataset_checks.csv'
AUDIT_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
df_clean = pd.read_csv(CLEANED_CSV)
X_train = pd.read_csv(SPLITS_DIR / 'train' / 'X_train.csv')
X_val = pd.read_csv(SPLITS_DIR / 'validation' / 'X_val.csv')
X_test = pd.read_csv(SPLITS_DIR / 'test' / 'X_test.csv')

### 2. Direct Identifier Redaction Verification

In [ ]:
forbidden_ids = {'charity_patient_no', 'surname', 'first_name', 'middle_name', 'contact_no', 'address', 'birthday', 'philhealth_no', 'email'}
leakage_cols = {'status', 'approved_financial_assistance', 'remaining_medical_expenses', 'social_worker_assessment_rating', 'relevant_social_worker_findings', 'assessment_notes', 'eligibility'}
id_hits = sorted(forbidden_ids.intersection(set(X_train.columns)))
leak_hits = sorted(leakage_cols.intersection(set(X_train.columns)))
checks = [
    {'check_name': 'no_direct_identifiers_in_features', 'category': 'privacy', 'expected_rule': '0 forbidden columns', 'observed_result': f'{len(id_hits)} found', 'status': 'PASS' if not id_hits else 'FAIL'},
    {'check_name': 'no_target_leakage_in_features', 'category': 'leakage', 'expected_rule': '0 post-decision/target columns', 'observed_result': f'{len(leak_hits)} found', 'status': 'PASS' if not leak_hits else 'FAIL'},
]

### 3. Schema and Type Invariant Checks

In [ ]:
total_split_rows = len(X_train) + len(X_val) + len(X_test)
checks.append({
    'check_name': 'split_row_conservation', 'category': 'integrity',
    'expected_rule': f'train+val+test == {len(df_clean)}',
    'observed_result': str(total_split_rows),
    'status': 'PASS' if total_split_rows == len(df_clean) else 'FAIL',
})
checks.append({
    'check_name': 'non_negative_income_in_train', 'category': 'range',
    'expected_rule': 'monthly_family_income >= 0',
    'observed_result': str(int((X_train['monthly_family_income'].dropna() < 0).sum())) + ' negative rows',
    'status': 'PASS' if (X_train['monthly_family_income'].dropna() < 0).sum() == 0 else 'FAIL',
})

### 4. Export Validation Certificate

In [ ]:
checks_df = pd.DataFrame(checks)
checks_df.to_csv(AUDIT_CSV_PATH, index=False)
cert = {'phase': '02_data_preparation', 'status': 'APPROVED' if all(c['status'] == 'PASS' for c in checks) else 'REVIEW_REQUIRED', 'zero_leakage_verified': len(leak_hits) == 0, 'checks_passed': int((checks_df['status'] == 'PASS').sum()), 'total_checks': len(checks_df)}
(VALIDATED_DIR / 'preparation_validation_certificate.json').write_text(json.dumps(cert, indent=2), encoding='utf-8')
print(checks_df.to_string(index=False))

### Findings & Observations

- **Observations:** Document validation sign-off, confirming data integrity, zero PHI leakage, and readiness for feature engineering.
- **Data Integrity & Governance Notes:** Verified that no raw patient identifiers or unmasked confidential records are displayed.


### Completion Checklist

- [ ] Direct patient identifiers verified completely absent.
- [ ] Target distribution checked for class balance and encoding correctness.
- [ ] Schema invariants checked across all prepared features.
- [ ] Validation report serialized in `data/interim/validated/`.
